# Labor SCC Tables
all paper tables, pulse year 2025. produces `.tex` files for overleaf

versions:
- uninteracted: model_collapsed (avg iiasa+oecd), iiasa only, oecd only
- interacted: model_collapsed, iiasa only, oecd only (ssp3 only, both iams)
- frisch mixed variants: oecd only (ssp3 only)
- panel III: RFF-SP probabilistic (pulse 2025, CO2_Fossil, euler_ramsey)s

In [1]:
pwd

'/project/cil/home_dirs/maiqi/repos/labor-code-release-2020/4_post_projection/4_scc'

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import os

PULSE_YEAR = 2025

for root in ["/project/cil", "/Volumes/project/cil", "/Volumes/cil"]:
    if os.path.isdir(root):
        break
else:
    raise FileNotFoundError("Cannot find project mount")

PROJECT = f"{root}/home_dirs/scadavidsanchez/projects/dscim-labor-2025-update2026"
BASE = f"{PROJECT}/results"
BASE_INT = f"{PROJECT}/results_interacted_model_v2"
BASE_FR1 = f"{PROJECT}/results_frisch_0p7_0p37"
BASE_FR2 = f"{PROJECT}/results_frisch_0p37_0p7"
OUT = "/project/cil/home_dirs/maiqi/repos/labor-code-release-2020/output/tables/scc_v6_full_precision"
os.makedirs(OUT, exist_ok=True)

DR = [0.015, 0.02, 0.025, 0.03, 0.05]

# dscim output is in 2019 USD; report in 2025 USD (BLS CPI-U annual average, 2019 -> 2025)
CPI_2019_TO_2025 = 321.943 / 255.657
# decimals shown in all tables; set to None for full precision
DECIMALS = 1

print(f"Pulse year: {PULSE_YEAR}")
print(f"CPI 2019->2025: {CPI_2019_TO_2025:.4f}, decimals: {DECIMALS}")
print(f"Output: {OUT}")

Pulse year: 2025
Output: /project/cil/home_dirs/maiqi/repos/labor-code-release-2020/output/tables/scc_v6_full_precision


# loaders and helpers

In [3]:
def _load(path, var):
    if not os.path.exists(path):
        print(f"  NOT FOUND: {os.path.basename(path)}")
        return None
    ds = xr.open_dataset(path)
    df = ds[var].to_dataframe().reset_index()
    ds.close()
    return df

def scc_mc(base, recipe="adding_up"):
    return _load(f"{base}/scc_output/AR6_ssp/labor/{PULSE_YEAR}/unmasked/{recipe}_constant_model_collapsed_eta2.0_rho0.0_scc.nc4", "scc")

def iqr_mc(base, recipe="adding_up"):
    return _load(f"{base}/scc_output_full_uncertainty/AR6_ssp/labor/{PULSE_YEAR}/unmasked/{recipe}_constant_model_collapsed_eta2.0_rho0.0_full_uncertainty_iqr.nc4", "full_uncertainty_iqr")

def scc_bm(base, model_name, recipe="adding_up"):
    path = f"{base}/scc_output_by_model/AR6_ssp/labor/{PULSE_YEAR}/unmasked/{recipe}_constant_eta2.0_rho0.0_scc.nc4"
    if not os.path.exists(path):
        print(f"  NOT FOUND: {os.path.basename(path)}")
        return None
    ds = xr.open_dataset(path)
    df = ds["scc"].sel(model=model_name).to_dataframe().reset_index()
    ds.close()
    return df

def iqr_bm(base, model_name, recipe="adding_up"):
    path = f"{base}/scc_output_full_uncertainty_by_model/AR6_ssp/labor/{PULSE_YEAR}/unmasked/{recipe}_constant_eta2.0_rho0.0_full_uncertainty_iqr.nc4"
    if not os.path.exists(path):
        print(f"  NOT FOUND: {os.path.basename(path)}")
        return None
    ds = xr.open_dataset(path)
    df = ds["full_uncertainty_iqr"].sel(model=model_name).to_dataframe().reset_index()
    ds.close()
    return df

def decomp_load(base, dtype, vname):
    return _load(f"{base}/scc_output_decomposition/{dtype}/AR6_ssp/labor/{PULSE_YEAR}/unmasked/adding_up_constant_model_collapsed_eta2.0_rho0.0_{vname}.nc4", vname)

def P(df, ssp="SSP3", fa="median_params"):
    if df is None or df.empty:
        return pd.DataFrame(columns=["rcp","discrate","scc"])
    m = (df.ssp==ssp) & (df.weitzman_parameter.astype(str)=="0") & (df.discrate.isin(DR))
    if "fair_aggregation" in df.columns:
        m = m & (df.fair_aggregation==fa)
    return df[m][["rcp","discrate","scc"]].sort_values(["rcp","discrate"]).reset_index(drop=True)

def Q(df, ssp="SSP3", qs=(0.05,0.95)):
    if df is None or df.empty:
        lo,hi = min(qs),max(qs)
        return pd.DataFrame(columns=["rcp","discrate",f"q{lo}",f"q{hi}"])
    lo,hi = min(qs),max(qs)
    m = (df.ssp==ssp) & (df.weitzman_parameter.astype(str)=="0") & (df.discrate.isin(DR))
    sub = df[m].copy()
    sub["quantile"] = sub["quantile"].round(4)
    sub = sub[sub["quantile"].isin([round(lo,4),round(hi,4)])]
    if sub.empty:
        return pd.DataFrame(columns=["rcp","discrate",f"q{lo}",f"q{hi}"])
    vc = [c for c in sub.columns if c not in ["rcp","discrate","quantile","ssp","weitzman_parameter","discount_type","model","gas","fair_aggregation"]][0]
    w = sub.pivot_table(index=["rcp","discrate"], columns="quantile", values=vc).reset_index()
    w = w.rename(columns={round(lo,4):f"q{lo}", round(hi,4):f"q{hi}"})
    return w.sort_values(["rcp","discrate"]).reset_index(drop=True)

def V(df, rcp, dr, col="scc"):
    if df is None or df.empty: return None
    r = df[(df.rcp==rcp)&(abs(df.discrate-dr)<0.001)]
    return r.iloc[0][col] if len(r)>0 else None

def N(v):
    return f"{v}" if DECIMALS is None else f"{v:.{DECIMALS}f}"

# raw 2019 USD in, formatted 2025 USD out; sym="$" for notebook display, default for latex
def D(v, sym=r"\$"):
    if v is None or pd.isna(v): return ""
    v = v * CPI_2019_TO_2025
    return f"-{sym}{N(abs(v))}" if v < 0 else f"{sym}{N(v)}"

def R(lo, hi, sym=r"\$"):
    if lo is None or hi is None: return ""
    return f"[{D(lo, sym)},{D(hi, sym)}]"

def wtex(name, lines):
    path = f"{OUT}/{name}.tex"
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print(f"  saved: {name}.tex")

<>:70: SyntaxWarning: invalid escape sequence '\$'
<>:74: SyntaxWarning: invalid escape sequence '\$'
<>:74: SyntaxWarning: invalid escape sequence '\$'
<>:70: SyntaxWarning: invalid escape sequence '\$'
<>:74: SyntaxWarning: invalid escape sequence '\$'
<>:74: SyntaxWarning: invalid escape sequence '\$'
/home/maiqi/ipykernel_452228/2011058562.py:70: SyntaxWarning: invalid escape sequence '\$'
  return f"\${v}"
/home/maiqi/ipykernel_452228/2011058562.py:74: SyntaxWarning: invalid escape sequence '\$'
  f = lambda v: f"-\${abs(v)}" if v<0 else f"\${v}"
/home/maiqi/ipykernel_452228/2011058562.py:74: SyntaxWarning: invalid escape sequence '\$'
  f = lambda v: f"-\${abs(v)}" if v<0 else f"\${v}"


# load all data

In [4]:
print("uninteracted: model_collapsed")
au_mc = scc_mc(BASE, "adding_up")
ra_mc = scc_mc(BASE, "risk_aversion")
iq_mc = iqr_mc(BASE)
dfunc_u = decomp_load(BASE, "damage_fn_uncertainty", "stat_uncertainty_iqr")
clim_u = decomp_load(BASE, "climate_uncertainty", "climate_uncertainty_iqr")

print("\nuninteracted: iiasa")
au_ii = scc_bm(BASE, "IIASA GDP", "adding_up")
ra_ii = scc_bm(BASE, "IIASA GDP", "risk_aversion")
iq_ii = iqr_bm(BASE, "IIASA GDP")

print("\nuninteracted: oecd")
au_oe = scc_bm(BASE, "OECD Env-Growth", "adding_up")
ra_oe = scc_bm(BASE, "OECD Env-Growth", "risk_aversion")
iq_oe = iqr_bm(BASE, "OECD Env-Growth")

print("\ninteracted: model_collapsed")
au_int_mc = scc_mc(BASE_INT, "adding_up")
ra_int_mc = scc_mc(BASE_INT, "risk_aversion")
iq_int_mc = iqr_mc(BASE_INT)
dfunc_i = decomp_load(BASE_INT, "damage_fn_uncertainty", "stat_uncertainty_iqr")
clim_i = decomp_load(BASE_INT, "climate_uncertainty", "climate_uncertainty_iqr")

print("\ninteracted: iiasa")
au_int_ii = scc_bm(BASE_INT, "IIASA GDP", "adding_up")
ra_int_ii = scc_bm(BASE_INT, "IIASA GDP", "risk_aversion")
iq_int_ii = iqr_bm(BASE_INT, "IIASA GDP")

print("\ninteracted: oecd")
au_int_oe = scc_bm(BASE_INT, "OECD Env-Growth", "adding_up")
ra_int_oe = scc_bm(BASE_INT, "OECD Env-Growth", "risk_aversion")
iq_int_oe = iqr_bm(BASE_INT, "OECD Env-Growth")

print("\nfrisch variants")
au_fr1 = scc_mc(BASE_FR1)
iq_fr1 = iqr_mc(BASE_FR1)
au_fr2 = scc_mc(BASE_FR2)
iq_fr2 = iqr_mc(BASE_FR2)

print("\ndone.")

uninteracted: model_collapsed

uninteracted: iiasa

uninteracted: oecd



interacted: model_collapsed



interacted: iiasa

interacted: oecd

frisch variants

done.


# table 5 generator

In [5]:
def make_table5(au, ra, iq, label, tag, ssp="SSP3"):
    p = P(au, ssp); r = Q(iq, ssp, (0.01,0.99)); ce = P(ra, ssp, "ce")
    if p.empty:
        print(f"table 5 {label}: no data, skipping")
        return
    rows = []
    for rcp in ["rcp85","rcp45"]:
        rl = "RCP 8.5" if rcp=="rcp85" else "RCP 4.5"
        for dr in DR:
            sv=V(p,rcp,dr); cv=V(ce,rcp,dr)
            rr=r[(r.rcp==rcp)&(abs(r.discrate-dr)<0.001)]
            lo=rr.iloc[0]["q0.01"] if len(rr)>0 else None
            hi=rr.iloc[0]["q0.99"] if len(rr)>0 else None
            rows.append({"RCP":rl,"dr":f"{dr*100}%","SCC":D(sv,"$"),"1-99%":R(lo,hi,"$"),"CE":D(cv,"$")})
    print(f"table 5: {label}")
    display(pd.DataFrame(rows))
    L = []
    L.append(r"\begin{table}[!htbp]")
    L.append(r"\centering")
    L.append(r"{\renewcommand{\arraystretch}{1.05}")
    L.append(r"\scriptsize{\begin{tabular}{>{\centering\arraybackslash}lccccc}")
    L.append(r"&&\\")
    L.append(r"\textbf{Discount rate} & $\delta = 1.5\%$ & $\delta = 2\%$ & $\delta = 2.5\%$ & $\delta = 3\%$ & $\delta = 5\%$\\")
    L.append(r"\hline")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{I. Partial SCC estimates}}\\")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{" + ssp + " " + label + r"}}\\")
    for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
        sv=" & ".join([D(V(p,rcp,dr)) for dr in DR])
        rv=" & ".join([R(V(r,rcp,dr,"q0.01"),V(r,rcp,dr,"q0.99")) for dr in DR])
        L.append(r"\hspace{0mm}\multirow{2}{*}{\textbf{" + rl + r"}} & " + sv + r" \\")
        L.append(r"\hspace{0mm} & " + rv + r" \\")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{II. Certainty equivalent}}\\")
    L.append(r"&&&&&\\")
    for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
        cv=" & ".join([D(V(ce,rcp,dr)) for dr in DR])
        L.append(r"\hspace{0mm}\textbf{" + rl + r"} & " + cv + r" \\")
    L.append(r"\hline")
    L.append(r"\end{tabular}}}")
    L.append(r"\caption{\textbf{Partial SCC for labor disutility} " + label + r". Pulse year " + str(PULSE_YEAR) + r".}")
    L.append(r"\label{tab:t5_" + tag + r"}")
    L.append(r"\end{table}")
    wtex(f"table5_{tag}", L)

# table 5: all versions

In [6]:
make_table5(au_mc, ra_mc, iq_mc, "uninteracted, model collapsed", "unint_mc")
make_table5(au_ii, ra_ii, iq_ii, "uninteracted, IIASA GDP", "unint_iiasa")
make_table5(au_oe, ra_oe, iq_oe, "uninteracted, OECD Env-Growth", "unint_oecd")
make_table5(au_int_mc, ra_int_mc, iq_int_mc, "interacted, model collapsed", "int_mc")
make_table5(au_int_ii, ra_int_ii, iq_int_ii, "interacted, IIASA GDP", "int_iiasa")
make_table5(au_int_oe, ra_int_oe, iq_int_oe, "interacted, OECD Env-Growth", "int_oecd")

table 5: uninteracted, model collapsed


,RCP,dr,SCC,1-99%,CE
0,RCP 8.5,1.5%,$18.361425015362535,"[-1.8931936920472532,161.7918096160733]",$31.885243099460013
1,RCP 8.5,2.0%,$10.647198995714682,"[-2.302482528993593,97.92183844771411]",$18.624421192728587
2,RCP 8.5,2.5%,$6.661600217395455,"[-2.4628133942358534,63.93961686750711]",$11.776599457486066
3,RCP 8.5,3.0%,$4.410000302718306,"[-2.5245261137858535,44.28581349942585]",$7.9127728940739175
4,RCP 8.5,5.0%,$1.1341999399915197,"[-2.3897452683541514,14.30391766011064]",$2.3094949290529962
5,RCP 4.5,1.5%,$11.37126529896557,"[-5.802566602679236,133.57212401253614]",$20.517658259406787
6,RCP 4.5,2.0%,$6.592657201061562,"[-5.210243525999914,77.43693607439188]",$11.861979598825027
7,RCP 4.5,2.5%,$4.11938407325411,"[-4.808227300933654,49.33135228939181]",$7.467775640635418
8,RCP 4.5,3.0%,$2.7134229122243223,"[-4.495041365589198,33.73407235442458]",$5.011113304721732
9,RCP 4.5,5.0%,$0.6254884979850882,"[-3.6043272033129883,11.292057955515551]",$1.4376021663947913


  saved: table5_unint_mc.tex


table 5: uninteracted, IIASA GDP


,RCP,dr,SCC,1-99%,CE
0,RCP 8.5,1.5%,$15.8017335614055,"[-1.9430164705496729,136.09457016818902]",$27.116428294549987
1,RCP 8.5,2.0%,$9.32314319816868,"[-2.318806495168108,83.43161081667131]",$16.137029981877934
2,RCP 8.5,2.5%,$5.9104924794112845,"[-2.4427333236597573,54.97348418286172]",$10.351831767307935
3,RCP 8.5,3.0%,$3.949399921843278,"[-2.4728208354894763,38.18534753681456]",$7.0298174625928675
4,RCP 8.5,5.0%,$1.0228655207150126,"[-2.300101825197951,12.126475410389386]",$2.087600274663764
5,RCP 4.5,1.5%,$9.955763146957336,"[-6.053598614281814,112.75871425015933]",$17.728396624564642
6,RCP 4.5,2.0%,$5.88145142203862,"[-5.321474401046947,66.50052672795894]",$10.452503403227135
7,RCP 4.5,2.5%,$3.727894307413211,"[-4.83611783179466,42.886948975312144]",$6.680945397774578
8,RCP 4.5,3.0%,$2.48060571262198,"[-4.467339218499497,29.589502427658633]",$4.532863939727358
9,RCP 4.5,5.0%,$0.5755970478184659,"[-3.5166264317732496,9.996157130389404]",$1.321677870584681


  saved: table5_unint_iiasa.tex
table 5: uninteracted, OECD Env-Growth


... (4 outputs left out)

,RCP,dr,SCC,1-99%,CE
0,RCP 8.5,1.5%,$40.52804240839369,"[0.8872629931221222,345.19916478728027]",$76.92229451543527
1,RCP 8.5,2.0%,$24.612528749861507,"[0.20049434411407271,202.1482775515748]",$54.47313168757853
2,RCP 8.5,2.5%,$16.319274105317977,"[-0.11336945995778389,127.0378226459297]",$38.91812058342204
3,RCP 8.5,3.0%,$11.595856706960317,"[-0.254181969299549,84.4277273546623]",$28.095932958277903
4,RCP 8.5,5.0%,$4.5755032890242555,"[-0.3505089345083776,23.313334360192048]",$8.764507750243995
5,RCP 4.5,1.5%,$25.017710515919962,"[-1.293813454168713,276.1396954726064]",$50.163249577913945
6,RCP 4.5,2.0%,$16.223375424345825,"[-1.0365498946675227,158.34528803292065]",$29.505949853278423
7,RCP 4.5,2.5%,$11.506446151893005,"[-0.889971515333817,99.23820990768458]",$19.154825993852988
8,RCP 4.5,3.0%,$8.712885610418343,"[-0.7963518222677554,66.67449397062275]",$13.429965700501059
9,RCP 4.5,5.0%,$4.137915523071029,"[-0.601939869781705,20.959482021221557]",$5.149942905167721


  saved: table5_int_iiasa.tex
table 5: interacted, OECD Env-Growth


,RCP,dr,SCC,1-99%,CE
0,RCP 8.5,1.5%,$52.532499366027,"[1.250976029179007,443.9854129292867]",$79.89879993146918
1,RCP 8.5,2.0%,$30.691527002099324,"[0.40017319262733053,253.14849746819215]",$55.92004403902582
2,RCP 8.5,2.5%,$19.696725886295518,"[0.013404353774230184,156.434057159335]",$39.574739392555415
3,RCP 8.5,3.0%,$13.626495462321108,"[-0.16529883745075727,102.96675568546812]",$28.372954303298226
4,RCP 8.5,5.0%,$5.033519721872319,"[-0.31511295748229945,28.846594435937696]",$8.861908960029908
5,RCP 4.5,1.5%,$32.78495742212072,"[-0.6666349900240569,365.6750335107109]",$63.725563295367635
6,RCP 4.5,2.0%,$20.191203614763083,"[-0.6090533062190806,201.49300869837657]",$36.300407626064235
7,RCP 4.5,2.5%,$13.724464190063522,"[-0.5781761452270081,122.36031479281323]",$22.921453512427934
8,RCP 4.5,3.0%,$10.046716861124894,"[-0.5622524681368561,80.39957178379863]",$15.706255540108755
9,RCP 4.5,5.0%,$4.408694722309853,"[-0.5054554358229701,24.465064832993477]",$5.6930966452501295


  saved: table5_int_oecd.tex


# table 6: wage flexibility + interacted
cols 1 to 5: uninteracted model_collapsed. col 6: interacted model_collapsed.
both use ssp3.

In [7]:
eps, rf = 0.5, 0.35
x_list = [0, 0.5, 1.5, 3.0, -2/3]
x_labels = ["Baseline","x=0.5","x=1.5","x=3","x=-0.67"]
sf_list = [1 - eps*x*rf for x in x_list]

pb = P(au_mc); rb = Q(iq_mc, qs=(0.05,0.95))
pi = P(au_int_mc); ri = Q(iq_int_mc, qs=(0.05,0.95))

rows = []
for rcp in ["rcp85","rcp45"]:
    rl = "RCP 8.5" if rcp=="rcp85" else "RCP 4.5"
    sv=V(pb,rcp,0.02)
    rr=rb[(rb.rcp==rcp)&(abs(rb.discrate-0.02)<0.001)]
    lo=rr.iloc[0]["q0.05"] if len(rr)>0 else None
    hi=rr.iloc[0]["q0.95"] if len(rr)>0 else None
    iv=V(pi,rcp,0.02)
    ir=ri[(ri.rcp==rcp)&(abs(ri.discrate-0.02)<0.001)]
    ilo=ir.iloc[0]["q0.05"] if len(ir)>0 else None
    ihi=ir.iloc[0]["q0.95"] if len(ir)>0 else None
    row = {"RCP": rl}
    for xl, sf in zip(x_labels, sf_list):
        row[xl] = f"{D(sv*sf)} {R(lo*sf,hi*sf)}" if sv else ""
    row["Interacted"] = f"{D(iv)} {R(ilo,ihi)}" if iv else "pending"
    rows.append(row)
print("table 6: SCC at delta=2%")
display(pd.DataFrame(rows))

L = []
L.append(r"\begin{landscape}")
L.append(r"\begin{table}[p]")
L.append(r"\centering")
L.append(r"\resizebox{\textheight}{!}{")
L.append(r"\begin{tabular}{llcccccc}")
L.append(r"\toprule")
L.append(r"& & \multicolumn{6}{c}{\textbf{Sensitivity to assumptions}} \\")
L.append(r"\cmidrule(lr){3-8}")
L.append(r"& & Baseline & \multicolumn{4}{c}{Short-term wage flexibility} & Income \\")
L.append(r"& & model & & & & & interaction \\")
L.append(r"\cmidrule(lr){3-3} \cmidrule(lr){4-7} \cmidrule(lr){8-8}")
L.append(r"& \textbf{Estimate} & (1) & (2) & (3) & (4) & (5) & (6) \\")
L.append(r"\cmidrule(lr){3-3} \cmidrule(lr){4-7} \cmidrule(lr){8-8}")
L.append(r"& & & $x = 0.5$ & $x = 1.5$ & $x = 3$ & $x = -0.67$ & \\")
L.append(r"& & & $r = 0.35$ & $r = 0.35$ & $r = 0.35$ & $r = 0.35$ & \\")
L.append(r"\addlinespace")
for rcp, rl, rn in [("rcp85","RCP8.5","(4)"),("rcp45","RCP4.5","(5)")]:
    sv=V(pb,rcp,0.02)
    rr=rb[(rb.rcp==rcp)&(abs(rb.discrate-0.02)<0.001)]
    lo=rr.iloc[0]["q0.05"] if len(rr)>0 else None
    hi=rr.iloc[0]["q0.95"] if len(rr)>0 else None
    iv=V(pi,rcp,0.02)
    ir=ri[(ri.rcp==rcp)&(abs(ri.discrate-0.02)<0.001)]
    ilo=ir.iloc[0]["q0.05"] if len(ir)>0 else None
    ihi=ir.iloc[0]["q0.95"] if len(ir)>0 else None
    sc=[D(sv*sf) if sv else "" for sf in sf_list]+[D(iv)]
    rc=[R(lo*sf,hi*sf) if lo else "" for sf in sf_list]+[R(ilo,ihi)]
    L.append(r"\multirow{2}{*}{" + rn + r"}")
    L.append(r"& Partial SCC (" + rl + r", $\delta=2\%$)")
    L.append(r"& " + " & ".join(sc) + r" \\")
    L.append(r"& & " + " & ".join(rc) + r" \\")
    L.append(r"\addlinespace")
L.append(r"\bottomrule")
L.append(r"\end{tabular}}")
L.append(r"\caption{\textbf{Sensitivity of estimates to assumptions.} SSP3, model collapsed, pulse year " + str(PULSE_YEAR) + r".}")
L.append(r"\label{tab:table6}")
L.append(r"\end{table}")
L.append(r"\end{landscape}")
wtex("table6_wage_flex", L)

table 6: SCC at delta=2%


,RCP,Baseline,x=0.5,x=1.5,x=3,x=-0.67,Interacted
0,RCP 8.5,"\$10.647198995714682 [-\$0.3253990325483568,\$...","\$9.715569083589648 [-\$0.29692661720037555,\$...","\$7.852309259339578 [-\$0.23998178650441315,\$...","\$5.057419522964475 [-\$0.1545645404604695,\$2...","\$11.889372211881396 [-\$0.36336225301233177,\...","\$27.555805369522986 [\$2.5988160705739225,\$1..."
1,RCP 4.5,"\$6.592657201061562 [-\$3.375661795470303,\$41...","\$6.015799695968675 [-\$3.0802913883666516,\$3...","\$4.862084685782902 [-\$2.4895505741593484,\$3...","\$3.1315121705042426 [-\$1.6034393528483941,\$...","\$7.361800541185411 [-\$3.7694890049418386,\$4...","\$18.142079955280323 [\$0.6923586949971866,\$9..."


  saved: table6_wage_flex.tex


# table 6 frisch: frisch elasticity sensitivity
cols 1 to 3: model_collapsed rescaled. cols 4 to 5: frisch mixed variants (oecd only, ssp3).
note: cols 4-5 are oecd only because maiqi only ran those for high iam.

In [8]:
pfr1 = P(au_fr1) if au_fr1 is not None else pd.DataFrame(columns=["rcp","discrate","scc"])
rfr1 = Q(iq_fr1, qs=(0.05,0.95)) if iq_fr1 is not None else pd.DataFrame()
pfr2 = P(au_fr2) if au_fr2 is not None else pd.DataFrame(columns=["rcp","discrate","scc"])
rfr2 = Q(iq_fr2, qs=(0.05,0.95)) if iq_fr2 is not None else pd.DataFrame()

p0 = P(au_mc); r0 = Q(iq_mc, qs=(0.05,0.95))
fsf = [1.0, 0.5/0.37, 0.5/0.7]

rows = []
for rcp in ["rcp85","rcp45"]:
    rl = "RCP 8.5" if rcp=="rcp85" else "RCP 4.5"
    sv=V(p0,rcp,0.02)
    rr=r0[(r0.rcp==rcp)&(abs(r0.discrate-0.02)<0.001)]
    lo=rr.iloc[0]["q0.05"] if len(rr)>0 else None
    hi=rr.iloc[0]["q0.95"] if len(rr)>0 else None
    row = {"RCP": rl}
    for sf, lbl in zip(fsf, ["e=0.5/0.5","e=0.37/0.37","e=0.7/0.7"]):
        row[lbl] = f"{D(sv*sf)} {R(lo*sf,hi*sf)}" if sv else ""
    v4=V(pfr1,rcp,0.02) if not pfr1.empty else None
    r4=rfr1[(rfr1.rcp==rcp)&(abs(rfr1.discrate-0.02)<0.001)] if not rfr1.empty else pd.DataFrame()
    lo4=r4.iloc[0]["q0.05"] if len(r4)>0 else None; hi4=r4.iloc[0]["q0.95"] if len(r4)>0 else None
    row["eH=0.7/eL=0.37"] = f"{D(v4)} {R(lo4,hi4)}" if v4 else "pending"
    v5=V(pfr2,rcp,0.02) if not pfr2.empty else None
    r5=rfr2[(rfr2.rcp==rcp)&(abs(rfr2.discrate-0.02)<0.001)] if not rfr2.empty else pd.DataFrame()
    lo5=r5.iloc[0]["q0.05"] if len(r5)>0 else None; hi5=r5.iloc[0]["q0.95"] if len(r5)>0 else None
    row["eH=0.37/eL=0.7"] = f"{D(v5)} {R(lo5,hi5)}" if v5 else "pending"
    rows.append(row)
print("table 6 frisch: SCC at delta=2%")
display(pd.DataFrame(rows))

L = []
L.append(r"\begin{landscape}")
L.append(r"\begin{table}[p]")
L.append(r"\centering")
L.append(r"\resizebox{\textheight}{!}{%")
L.append(r"\begin{tabular}{llccccc}")
L.append(r"\toprule")
L.append(r"& & \multicolumn{5}{c}{\textbf{Sensitivity to assumptions}} \\")
L.append(r"\cmidrule(lr){3-7}")
L.append(r"& & Baseline & \multicolumn{4}{c}{Varying Frisch elasticity} \\")
L.append(r"\cmidrule(lr){3-3} \cmidrule(lr){4-7}")
L.append(r"& \textbf{Estimate} & (1) & (2) & (3) & (4) & (5) \\")
L.append(r"\cmidrule(lr){3-3} \cmidrule(lr){4-7}")
L.append(r"& &$\varepsilon_H = 0.5$ & $\varepsilon_H = 0.37$ & $\varepsilon_H = 0.7$ & $\varepsilon_H = 0.7$ & $\varepsilon_H = 0.37$ \\")
L.append(r"& & $\varepsilon_L = 0.5$& $\varepsilon_L = 0.37$ & $\varepsilon_L = 0.7$ & $\varepsilon_L = 0.37$ & $\varepsilon_L = 0.7$ \\")
L.append(r"\addlinespace")
for rcp, rl, rn in [("rcp85","RCP8.5","(4)"),("rcp45","RCP4.5","(5)")]:
    sv=V(p0,rcp,0.02)
    rr=r0[(r0.rcp==rcp)&(abs(r0.discrate-0.02)<0.001)]
    lo=rr.iloc[0]["q0.05"] if len(rr)>0 else None
    hi=rr.iloc[0]["q0.95"] if len(rr)>0 else None
    sc=[D(sv*sf) if sv else "" for sf in fsf]
    rc=[R(lo*sf,hi*sf) if lo else "" for sf in fsf]
    for pf, rf_df in [(pfr1,rfr1),(pfr2,rfr2)]:
        vx=V(pf,rcp,0.02) if not pf.empty else None
        rx=rf_df[(rf_df.rcp==rcp)&(abs(rf_df.discrate-0.02)<0.001)] if not rf_df.empty else pd.DataFrame()
        sc.append(D(vx))
        rc.append(R(rx.iloc[0]["q0.05"] if len(rx)>0 else None, rx.iloc[0]["q0.95"] if len(rx)>0 else None))
    L.append(r"\multirow{2}{*}{" + rn + r"}")
    L.append(r"& Partial SCC (" + rl + r", $\delta=2\%$)")
    L.append(r"& " + " & ".join(sc) + r" \\")
    L.append(r"& & " + " & ".join(rc) + r" \\")
    L.append(r"\addlinespace")
L.append(r"\bottomrule")
L.append(r"\end{tabular}}")
L.append(r"\caption{\textbf{Sensitivity: Frisch elasticity.} SSP3, cols 1-3 model collapsed, cols 4-5 OECD only, pulse year " + str(PULSE_YEAR) + r".}")
L.append(r"\label{tab:t6xr}")
L.append(r"\end{table}")
L.append(r"\end{landscape}")
wtex("table6_frisch", L)

table 6 frisch: SCC at delta=2%


,RCP,e=0.5/0.5,e=0.37/0.37,e=0.7/0.7,eH=0.7/eL=0.37,eH=0.37/eL=0.7
0,RCP 8.5,"\$10.647198995714682 [-\$0.3253990325483568,\$...","\$14.388106750965786 [-\$0.4397284223626443,\$...","\$7.605142139796201 [-\$0.2324278803916834,\$4...","\$11.6888284187087 [-\$0.42653239422755723,\$5...","\$12.417073242204804 [-\$0.7190637378146975,\$..."
1,RCP 4.5,"\$6.592657201061562 [-\$3.375661795470303,\$41...","\$8.90899621765076 [-\$4.561705129013923,\$56....","\$4.709040857901116 [-\$2.4111869967645023,\$2...","\$8.121772429215993 [-\$3.09337694489743,\$46....","\$6.58169722566785 [-\$4.34292213746969,\$48.8..."


  saved: table6_frisch.tex


# table h.1: uncertainty decomposition

In [9]:
def make_h1(au, iq, df_iq, cl_iq, label, tag):
    ph=P(au); rfull=Q(iq,qs=(0.01,0.99))
    if ph.empty:
        print(f"table h.1 {label}: no data, skipping")
        return
    rdf=Q(df_iq,qs=(0.01,0.99)) if df_iq is not None else pd.DataFrame()
    rcl=Q(cl_iq,qs=(0.01,0.99)) if cl_iq is not None else pd.DataFrame()
    rows = []
    for rcp in ["rcp85","rcp45"]:
        rl = "RCP 8.5" if rcp=="rcp85" else "RCP 4.5"
        for dr in DR:
            sv=V(ph,rcp,dr)
            fl=rfull[(rfull.rcp==rcp)&(abs(rfull.discrate-dr)<0.001)]
            dl=rdf[(rdf.rcp==rcp)&(abs(rdf.discrate-dr)<0.001)] if not rdf.empty else pd.DataFrame()
            cl=rcl[(rcl.rcp==rcp)&(abs(rcl.discrate-dr)<0.001)] if not rcl.empty else pd.DataFrame()
            rows.append({"RCP":rl,"dr":f"{dr*100}%","Point":D(sv,"$"),
                "Full":R(fl.iloc[0]['q0.01'],fl.iloc[0]['q0.99'],"$") if len(fl)>0 else "",
                "DmgFn":R(dl.iloc[0]['q0.01'],dl.iloc[0]['q0.99'],"$") if len(dl)>0 else "",
                "Climate":R(cl.iloc[0]['q0.01'],cl.iloc[0]['q0.99'],"$") if len(cl)>0 else ""})
    print(f"table h.1: {label}")
    display(pd.DataFrame(rows))
    L = []
    L.append(r"\begin{table}[!htbp]")
    L.append(r"\centering")
    L.append(r"{\renewcommand{\arraystretch}{1.05}")
    L.append(r"\scriptsize{\begin{tabular}{>{\centering\arraybackslash}lccccc}")
    L.append(r"&&\\")
    L.append(r"\textbf{Discount rate} & $\delta = 1.5\%$ & $\delta = 2\%$ & $\delta = 2.5\%$ & $\delta = 3\%$ & $\delta = 5\%$\\")
    L.append(r"\hline")
    for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
        L.append(r"&&&&&\\")
        pl=" & ".join([D(V(ph,rcp,dr)) for dr in DR])
        L.append(r"\textbf{" + rl + r"} & " + pl + r" \\")
        fl=" & ".join([R(V(rfull,rcp,dr,"q0.01"),V(rfull,rcp,dr,"q0.99")) for dr in DR])
        L.append(r"\quad Full uncertainty & " + fl + r" \\")
        if not rdf.empty:
            dl=" & ".join([R(V(rdf,rcp,dr,"q0.01"),V(rdf,rcp,dr,"q0.99")) for dr in DR])
            L.append(r"\quad Damage fn only & " + dl + r" \\")
        if not rcl.empty:
            cl=" & ".join([R(V(rcl,rcp,dr,"q0.01"),V(rcl,rcp,dr,"q0.99")) for dr in DR])
            L.append(r"\quad Climate sens only & " + cl + r" \\")
    L.append(r"\hline")
    L.append(r"\end{tabular}}}")
    L.append(r"\caption{\textbf{Uncertainty decomposition} " + label + r". Pulse year " + str(PULSE_YEAR) + r".}")
    L.append(r"\label{tab:h1_" + tag + r"}")
    L.append(r"\end{table}")
    wtex(f"table_h1_{tag}", L)

make_h1(au_mc, iq_mc, dfunc_u, clim_u, "uninteracted, model collapsed", "uninteracted")
make_h1(au_int_mc, iq_int_mc, dfunc_i, clim_i, "interacted, model collapsed", "interacted")

table h.1: uninteracted, model collapsed


,RCP,dr,Point,Full,DmgFn,Climate
0,RCP 8.5,1.5%,$18.361425015362535,"[-1.8931936920472532,161.7918096160733]","[2.440328628832153,35.125973999950254]","[2.6847485205116657,156.22750376730255]"
1,RCP 8.5,2.0%,$10.647198995714682,"[-2.302482528993593,97.92183844771411]","[-0.3878852777311287,22.005866152949224]","[1.5439578540439247,94.92048421165164]"
2,RCP 8.5,2.5%,$6.661600217395455,"[-2.4628133942358534,63.93961686750711]","[-1.6560606106958373,15.053127109386281]","[0.9471461813719978,62.25247905292898]"
3,RCP 8.5,3.0%,$4.410000302718306,"[-2.5245261137858535,44.28581349942585]","[-2.242943376378761,11.010598730760986]","[0.6036826720867017,43.2594068387096]"
4,RCP 8.5,5.0%,$1.1341999399915197,"[-2.3897452683541514,14.30391766011064]","[-2.62145681477134,4.713791347783196]","[0.09560352372348224,14.371826539909746]"
5,RCP 4.5,1.5%,$11.37126529896557,"[-5.802566602679236,133.57212401253614]","[-5.558605606251288,29.34849494108466]","[1.901822129290258,123.10330727187021]"
6,RCP 4.5,2.0%,$6.592657201061562,"[-5.210243525999914,77.43693607439188]","[-5.295452266487557,18.943235387008915]","[1.0914732622702115,70.72137020930217]"
7,RCP 4.5,2.5%,$4.11938407325411,"[-4.808227300933654,49.33135228939181]","[-4.980999193598583,13.386500534188508]","[0.660213491849887,44.572610351220455]"
8,RCP 4.5,3.0%,$2.7134229122243223,"[-4.495041365589198,33.73407235442458]","[-4.677599897827955,10.109796193865451]","[0.4099360744899727,30.248102147616024]"
9,RCP 4.5,5.0%,$0.6254884979850882,"[-3.6043272033129883,11.292057955515551]","[-3.7352799440941538,4.785939114158533]","[0.02600511054894368,9.698621339833926]"


  saved: table_h1_uninteracted.tex


table h.1: interacted, model collapsed


,RCP,dr,Point,Full,DmgFn,Climate
0,RCP 8.5,1.5%,$46.284140142624416,"[1.0615892232484643,393.43940390897734]","[5.712762322103325,93.48101714486532]","[7.636487537278709,361.8991410551087]"
1,RCP 8.5,2.0%,$27.555805369522986,"[0.29700154970172254,227.1967319651189]","[2.6782420115664918,56.209504884575935]","[4.990798759297735,210.6042407301258]"
2,RCP 8.5,2.5%,$17.96791378097533,"[-0.04980438590606516,141.59341396289338]","[1.2542058520791546,36.94726187542237]","[3.555547826101054,131.5801690252241]"
3,RCP 8.5,3.0%,$12.593417530688344,"[-0.2086891084030926,93.61118230636275]","[0.5394062911512265,26.060864532636394]","[2.7081232037319385,87.25461224934159]"
4,RCP 8.5,5.0%,$4.803397355691221,"[-0.3326532394906004,26.154350416308613]","[-0.2278694324693485,10.11619952109226]","[1.2965097319555536,24.819355042386157]"
5,RCP 4.5,1.5%,$28.73444781869233,"[-0.9286353124773247,319.60142427437324]","[0.43718627892408635,70.89485103150086]","[5.524759098328064,286.623753910405]"
6,RCP 4.5,2.0%,$18.142079955280323,"[-0.7858927526516482,179.51566711948513]","[-0.19182575160507234,44.56708472117076]","[3.907395346680339,159.52714247963735]"
7,RCP 4.5,2.5%,$12.588292373483522,"[-0.7091031692025509,110.77823916550348]","[-0.4504930262456823,30.750685738030114]","[2.9582930423720524,98.0003163513258]"
8,RCP 4.5,3.0%,$9.367763827978788,"[-0.6620972018687734,73.56621258834103]","[-0.5558350766706158,22.757763900154295]","[2.3668055331728346,64.68012220893797]"
9,RCP 4.5,5.0%,$4.272545767887583,"[-0.5488928977971647,22.76129713299575]","[-0.5752179416952979,10.269904584083982]","[1.2630200323783753,19.68954223248899]"


  saved: table_h1_interacted.tex


# table h.2: alternative ssps
note: interacted model only has ssp3 (wage-levels files for ssp2/ssp4 were not produced).
so table h.2 for interacted cannot be generated.

In [10]:
def make_h2(au, ra, iq, label, tag):
    rows = []
    for ssp in ["SSP2","SSP3","SSP4"]:
        ps=P(au,ssp=ssp)
        if ps.empty:
            print(f"  {ssp}: no data, skipping")
            continue
        rs=Q(iq,ssp=ssp,qs=(0.01,0.99)); cs=P(ra,ssp=ssp,fa="ce")
        for rcp in ["rcp85","rcp45"]:
            rl = "RCP 8.5" if rcp=="rcp85" else "RCP 4.5"
            for dr in DR:
                sv=V(ps,rcp,dr); cv=V(cs,rcp,dr)
                rr=rs[(rs.rcp==rcp)&(abs(rs.discrate-dr)<0.001)]
                rows.append({"SSP":ssp,"RCP":rl,"dr":f"{dr*100}%",
                    "SCC":D(sv,"$"),"1-99%":R(rr.iloc[0]['q0.01'],rr.iloc[0]['q0.99'],"$") if len(rr)>0 else "",
                    "CE":D(cv,"$")})
    if not rows:
        print(f"table h.2 {label}: no data for any ssp, skipping")
        return
    print(f"table h.2: {label}")
    display(pd.DataFrame(rows))
    L = []
    L.append(r"\begin{table}[!htbp]")
    L.append(r"\centering")
    L.append(r"{\renewcommand{\arraystretch}{1.05}")
    L.append(r"\scriptsize{\begin{tabular}{>{\centering\arraybackslash}lccccc}")
    L.append(r"&&\\")
    L.append(r"\textbf{Discount rate} & $\delta = 1.5\%$ & $\delta = 2\%$ & $\delta = 2.5\%$ & $\delta = 3\%$ & $\delta = 5\%$\\")
    L.append(r"\hline")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{I. Partial SCC estimates}}\\")
    for ssp in ["SSP2","SSP3","SSP4"]:
        ps=P(au,ssp=ssp)
        if ps.empty: continue
        L.append(r"&&&&&\\")
        L.append(r"&\multicolumn{5}{c}{\textbf{" + ssp + r"}}\\")
        rs=Q(iq,ssp=ssp,qs=(0.01,0.99))
        for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
            sv=" & ".join([D(V(ps,rcp,dr)) for dr in DR])
            rv=" & ".join([R(V(rs,rcp,dr,"q0.01"),V(rs,rcp,dr,"q0.99")) for dr in DR])
            L.append(r"\hspace{0mm}\multirow{2}{*}{\textbf{" + rl + r"}} & " + sv + r" \\")
            L.append(r"\hspace{0mm} & " + rv + r" \\")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{II. Certainty equivalent}}\\")
    for ssp in ["SSP2","SSP3","SSP4"]:
        cs=P(ra,ssp=ssp,fa="ce")
        if cs.empty: continue
        L.append(r"&&&&&\\")
        L.append(r"&\multicolumn{5}{c}{\textbf{" + ssp + r"}}\\")
        for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
            cv=" & ".join([D(V(cs,rcp,dr)) for dr in DR])
            L.append(r"\hspace{0mm}\textbf{" + rl + r"} & " + cv + r" \\")
    L.append(r"\hline")
    L.append(r"\end{tabular}}}")
    L.append(r"\caption{\textbf{Alternative socioeconomic scenarios} " + label + r". Pulse year " + str(PULSE_YEAR) + r".}")
    L.append(r"\label{tab:h2_" + tag + r"}")
    L.append(r"\end{table}")
    wtex(f"table_h2_{tag}", L)

make_h2(au_mc, ra_mc, iq_mc, "uninteracted, model collapsed", "uninteracted")
make_h2(au_int_mc, ra_int_mc, iq_int_mc, "interacted, model collapsed", "interacted")

table h.2: uninteracted, model collapsed


,SSP,RCP,dr,SCC,1-99%,CE
0,SSP2,RCP 8.5,1.5%,$48.93937911912521,"[-1.4536833346023998,382.4332527519869]",$82.24429407720231
1,SSP2,RCP 8.5,2.0%,$23.56120073552644,"[-2.224149683640713,187.97485157093254]",$39.69583288260761
2,SSP2,RCP 8.5,2.5%,$12.751795512157585,"[-2.497691302684704,103.92561088792559]",$21.54023924941588
3,SSP2,RCP 8.5,3.0%,$7.621906859302184,"[-2.623253025098754,63.52665005485609]",$12.921014964114036
4,SSP2,RCP 8.5,5.0%,$1.7115998949376796,"[-2.5527583848212005,15.54620815085219]",$3.048733625145144
5,SSP2,RCP 4.5,1.5%,$31.149536889081432,"[-7.188861816641532,368.49924254108043]",$56.784356337475494
6,SSP2,RCP 4.5,2.0%,$15.114602519020183,"[-5.820885770564338,170.79783156680955]",$26.866792903364363
7,SSP2,RCP 4.5,2.5%,$8.293532387124396,"[-5.146940534578694,90.0713785983726]",$14.4261548656126
8,SSP2,RCP 4.5,3.0%,$5.04033844561348,"[-4.732607234330879,53.41890203401396]",$8.644625588390198
9,SSP2,RCP 4.5,5.0%,$1.1805511659072223,"[-3.7813387576862145,13.596257111221817]",$2.10053702605317


  saved: table_h2_uninteracted.tex
  SSP2: no data, skipping
  SSP4: no data, skipping
table h.2: interacted, model collapsed


,SSP,RCP,dr,SCC,1-99%,CE
0,SSP3,RCP 8.5,1.5%,$46.284140142624416,"[1.0615892232484643,393.43940390897734]",$33.392137883117066
1,SSP3,RCP 8.5,2.0%,$27.555805369522986,"[0.29700154970172254,227.1967319651189]",$25.24211814589334
2,SSP3,RCP 8.5,2.5%,$17.96791378097533,"[-0.04980438590606516,141.59341396289338]",$19.311942688555664
3,SSP3,RCP 8.5,3.0%,$12.593417530688344,"[-0.2086891084030926,93.61118230636275]",$14.965983845251419
4,SSP3,RCP 8.5,5.0%,$4.803397355691221,"[-0.3326532394906004,26.154350416308613]",$6.2046300755266355
5,SSP3,RCP 4.5,1.5%,$28.73444781869233,"[-0.9286353124773247,319.60142427437324]",$56.691289488705074
6,SSP3,RCP 4.5,2.0%,$18.142079955280323,"[-0.7858927526516482,179.51566711948513]",$32.805322376199435
7,SSP3,RCP 4.5,2.5%,$12.588292373483522,"[-0.7091031692025509,110.77823916550348]",$20.997717519039313
8,SSP3,RCP 4.5,3.0%,$9.367763827978788,"[-0.6620972018687734,73.56621258834103]",$14.55028598792029
9,SSP3,RCP 4.5,5.0%,$4.272545767887583,"[-0.5488928977971647,22.76129713299575]",$5.420364576320218


  saved: table_h2_interacted.tex


# panel III: RFF probabilistic approach
pulse year 2025, CO2_Fossil only, euler_ramsey discounting with 10k RFF-SP draws.
uses adding_up mean (risk_aversion gives nearly identical values at these low etas).

In [11]:
# panel III: RFF probabilistic (CO2_Fossil, weitzman=0)
# uses adding_up mean (risk_aversion gives nearly identical values at these low etas)

print(f"Panel III: RFF-SP probabilistic approach (pulse {PULSE_YEAR})")
eta_rho = [
    (1.016010255, 9.149608e-05, "1.5%"),
    (1.244459066, 0.00197263997, "2.0%"),
    (1.421158116, 0.00461878399, "2.5%"),
]
rff_vals = {}
for eta, rho, label in eta_rho:
    path = f"{BASE}/scc_output/rff_results/labor/{PULSE_YEAR}/unmasked/adding_up_euler_ramsey_eta{eta}_rho{rho}_uncollapsed_sccs.nc4"
    if not os.path.exists(path):
        print(f"  {label}: NOT FOUND")
        continue
    ds = xr.open_dataset(path)
    scc = ds["uncollapsed_sccs"].sel(gas="CO2_Fossil", weitzman_parameter="0")
    rff_vals[label] = float(scc.mean())  # 2019 USD; D() converts at output
    print(f"  {label}: {D(rff_vals[label], '$')}")
    ds.close()

if not rff_vals:
    print("  no RFF results found, skipping Panel III")

Panel III: RFF-SP probabilistic approach (pulse 2025)


  1.5%: $45.228644591912385
  2.0%: $24.562771553968393
  2.5%: $14.270797936800562


In [12]:
# add Panel III to table5_unint_mc.tex
if rff_vals:
    tex_path = f"{OUT}/table5_unint_mc.tex"
    with open(tex_path) as f:
        tex = f.read()

    v15 = D(rff_vals.get("1.5%"))
    v20 = D(rff_vals.get("2.0%"))
    v25 = D(rff_vals.get("2.5%"))

    panel3_lines = [
        r'&&&&&\\',
        r'&\multicolumn{5}{c}{\textbf{III. Partial SCC estimates}}\\',
        r'&\multicolumn{5}{c}{\textbf{(U.S. government approach, certainty equivalent)}}\\',
        r'&&&&&\\',
        r'\hspace{0mm}\textbf{Probabilistic scenarios} & ' + v15 + ' & ' + v20 + ' & ' + v25 + r' & & \\',
        r'\hspace{0mm}\textbf{(Rennert et al. 2022b)} & & & & & \\',
    ]
    panel3 = '\n'.join(panel3_lines)

    # insert before the last \hline
    tex = tex.replace(r'\hline' + '\n' + r'\end{tabular}', panel3 + '\n' + r'\hline' + '\n' + r'\end{tabular}')

    # update caption to note Panel III pulse year
    tex = tex.replace(
        f'Pulse year {PULSE_YEAR}.',
        f'Panels I-II: pulse year {PULSE_YEAR}. Panel III: pulse year {PULSE_YEAR} (RFF-SP).'
    )

    with open(tex_path, 'w') as f:
        f.write(tex)
    print(f'Panel III added to table5_unint_mc.tex')
    print(f'Values (2025 USD): { {k: D(v, "$") for k, v in rff_vals.items()} }')
else:
    print('no RFF values, Panel III not added')

Panel III added to table5_unint_mc.tex
Values: {'1.5%': 45.228644591912385, '2.0%': 24.562771553968393, '2.5%': 14.270797936800562}


<>:7: SyntaxWarning: invalid escape sequence '\$'
<>:8: SyntaxWarning: invalid escape sequence '\$'
<>:9: SyntaxWarning: invalid escape sequence '\$'
<>:7: SyntaxWarning: invalid escape sequence '\$'
<>:8: SyntaxWarning: invalid escape sequence '\$'
<>:9: SyntaxWarning: invalid escape sequence '\$'
/home/maiqi/ipykernel_452228/3931575216.py:7: SyntaxWarning: invalid escape sequence '\$'
  v15 = f'\${rff_vals.get("1.5%",0)}'
/home/maiqi/ipykernel_452228/3931575216.py:8: SyntaxWarning: invalid escape sequence '\$'
  v20 = f'\${rff_vals.get("2.0%",0)}'
/home/maiqi/ipykernel_452228/3931575216.py:9: SyntaxWarning: invalid escape sequence '\$'
  v25 = f'\${rff_vals.get("2.5%",0)}'


# cleanup old files from previous runs

In [13]:
import glob
old_files = ["table5_int_model_collapsed.tex", "table5_int_iiasa.tex", "table5_int_oecd.tex",
             "table5_interacted.tex", "table5_model_collapsed.tex", "table5_iiasa.tex", 
             "table5_oecd.tex", "table5_unint_model_collapsed.tex", "table5_unint_oecd.tex",
             "table5.tex", "table6.tex", "table_h1_interacted.tex", "table_h2.tex"]
for f in old_files:
    p = f"{OUT}/{f}"
    if os.path.exists(p):
        os.remove(p)
        print(f"  removed: {f}")

  removed: table5_int_iiasa.tex
  removed: table5_int_oecd.tex
  removed: table5_unint_oecd.tex
  removed: table_h1_interacted.tex


# summary

In [14]:
print(f"output: {OUT}")
for f in sorted(os.listdir(OUT)):
    if f.endswith(".tex"):
        print(f"  {f}")

output: /project/cil/home_dirs/maiqi/repos/labor-code-release-2020/output/tables/scc_v6_full_precision
  table5_int_mc.tex
  table5_unint_iiasa.tex
  table5_unint_mc.tex
  table6_frisch.tex
  table6_wage_flex.tex
  table_h1_uninteracted.tex
  table_h2_interacted.tex
  table_h2_uninteracted.tex


In [15]:
pwd

'/project/cil/home_dirs/maiqi/repos/labor-code-release-2020/4_post_projection/4_scc'